<a href="https://colab.research.google.com/github/jennifersolano308-cyber/AAI2025/blob/main/Part_2_Code_Generation_with_ReACT_Prompting_Goal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q openai

In [10]:
import os, re, sys, subprocess, tempfile, time
from openai import OpenAI

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

client = OpenAI(api_key=os.environ["GEMINI_API_KEY"],
                base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

MODEL = "gemini-3.8-flash"   # replacement named in Google's 404 message
MAX_CYCLES = 5

def pick_working_model():
    """If MODEL is rejected, pick the first available flash model."""
    global MODEL
    try:
        client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=5)
        print("Using model:", MODEL)
    except Exception as e:
        if "404" in str(e) or "NOT_FOUND" in str(e):
            ids = [m.id.replace("models/", "") for m in client.models.list()]
            flash = [i for i in ids if "flash" in i and "image" not in i and "tts" not in i]
            if not flash:
                raise RuntimeError(f"No flash model found. Available: {ids}")
            MODEL = sorted(flash)[-1]
            print("Switched to model:", MODEL)
        else:
            raise

pick_working_model()

Using model: gemini-3.8-flash


In [11]:
REACT_SYSTEM = """You are a Python coding agent that works in a ReACT loop (Reason, Act, Observe).

Work in these stages, one cycle per reply:

THOUGHT: Reason briefly (max 4 sentences). On the first cycle, plan the approach and list the edge cases.
On later cycles, explain what the OBSERVATION shows went wrong and how you will fix it.

ACTION: Output ONE complete, runnable Python script inside a single ```python code block.
The script must define the requested function AND include assert-based tests that cover
normal input and every edge case, then print "ALL TESTS PASSED" at the end if they pass.

After each ACTION, the system will run your script and reply with:
OBSERVATION: <stdout/stderr from the run>

Repeat THOUGHT -> ACTION until the OBSERVATION shows "ALL TESTS PASSED" with no errors.
Then reply with exactly:
FINAL ANSWER: <one sentence on what the code does and what you fixed>

Rules:
- Use only the Python standard library.
- Never claim success unless an OBSERVATION shows it.
- Do not output more than one code block per reply.
- Do not include explanations outside the THOUGHT / ACTION / FINAL ANSWER format."""

TASK = """Write a function summarize_sales(csv_text: str) -> dict.

Input: CSV text with the header "date,product,quantity,price". Example row: 2024-03-01,Widget,3,9.99

Output: a dict with these keys:
  "revenue_by_product": dict mapping product -> total revenue (quantity * price), rounded to 2 decimals,
                        ordered from highest to lowest revenue
  "top_product": the product with the highest revenue, or None if there are no valid rows
  "skipped_rows": number of rows skipped

Requirements:
- Skip (and count) malformed rows: wrong number of columns, non-numeric quantity or price,
  negative quantity, or negative price.
- Product names should be trimmed of whitespace and compared case-insensitively
  (store them in the form first seen).
- Empty input or a header-only input returns {"revenue_by_product": {}, "top_product": None, "skipped_rows": 0}.
- Do not raise exceptions for bad data.
- Use only the standard library (csv, io, collections allowed).
- Include assert tests for: normal data, empty input, header-only input, malformed rows,
  case/whitespace differences in product names, and a tie in revenue."""

In [12]:
def run_code(code, timeout=15):
    """ACT: execute the generated script and return (success, observation)."""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code)
        path = f.name
    try:
        p = subprocess.run([sys.executable, path], capture_output=True,
                           text=True, timeout=timeout)
        output = (p.stdout + p.stderr).strip()
        return p.returncode == 0, output[-3000:] or "(no output)"
    except subprocess.TimeoutExpired:
        return False, f"Error: script timed out after {timeout} seconds."
    finally:
        os.remove(path)

def extract_code(reply):
    m = re.search(r"```python\s*(.*?)```", reply, re.DOTALL)
    return m.group(1).strip() if m else None

def ask_model(messages, retries=3):
    """Call the model, retrying on rate-limit errors (common on free tiers)."""
    for attempt in range(retries):
        try:
            return client.chat.completions.create(
                model=MODEL, temperature=0.2, messages=messages
            ).choices[0].message.content
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                print("Rate limited, waiting 20s...")
                time.sleep(20)
            else:
                raise

def react_loop(task, max_cycles=MAX_CYCLES):
    messages = [{"role": "system", "content": REACT_SYSTEM},
                {"role": "user", "content": task}]
    last_good_code, passed = None, False

    for cycle in range(1, max_cycles + 1):
        print(f"\n{'='*20} CYCLE {cycle} {'='*20}")
        reply = ask_model(messages)
        messages.append({"role": "assistant", "content": reply})
        print(reply)

        code = extract_code(reply)
        if code:
            ok, obs = run_code(code)
            print(f"\nOBSERVATION (exit ok={ok}):\n{obs}")
            if ok and "ALL TESTS PASSED" in obs:
                passed, last_good_code = True, code
            messages.append({"role": "user", "content": f"OBSERVATION: {obs}"})
        elif "FINAL ANSWER" in reply:
            if passed:
                print("\nLoop finished: tests passed.")
                return last_good_code
            messages.append({"role": "user", "content":
                "OBSERVATION: No successful test run has been shown yet. Provide an ACTION."})
        else:
            messages.append({"role": "user", "content":
                "OBSERVATION: Invalid format. Reply with THOUGHT then one ```python block."})

    print("\nReached max cycles.")
    return last_good_code

final_code = react_loop(TASK)


==================== CYCLE 1 ====================
ACTION:
```python
import csv
import io
from collections import OrderedDict


def summarize_sales(csv_text: str) -> dict:
    default_result = {
        "revenue_by_product": {},
        "top_product": None,
        "skipped_rows": 0,
    }
    if not csv_text or not csv_text.strip():
        return default_result

    f = io.StringIO(csv_text.strip())
    reader = csv.reader(f)

    try:
        header = next(reader)
    except StopIteration:
        return default_result

    # Check if first row is header
    # Header: date,product,quantity,price
    header_clean = [col.strip().lower() for col in header]
    if header_clean != ["date", "product", "quantity", "price"]:
        # If it doesn't match the expected header, return default with 0 skipped or treat as invalid
        # But per spec: "Input: CSV text with the header 'date,product,quantity,price'."
        # So we assume the first line is the header.
        pass

    product_n

In [13]:
print(final_code)
print("\n--- Independent re-run ---")
ok, out = run_code(final_code)
print(out)

import csv
import io
from collections import OrderedDict


def summarize_sales(csv_text: str) -> dict:
    default_result = {
        "revenue_by_product": {},
        "top_product": None,
        "skipped_rows": 0,
    }
    if not csv_text or not csv_text.strip():
        return default_result

    f = io.StringIO(csv_text.strip())
    reader = csv.reader(f)

    try:
        header = next(reader)
    except StopIteration:
        return default_result

    # Check if first row is header
    # Header: date,product,quantity,price
    header_clean = [col.strip().lower() for col in header]
    if header_clean != ["date", "product", "quantity", "price"]:
        # If it doesn't match the expected header, return default with 0 skipped or treat as invalid
        # But per spec: "Input: CSV text with the header 'date,product,quantity,price'."
        # So we assume the first line is the header.
        pass

    product_names = {}  # lower_name -> first_seen_name
    product_revenues = {} 

In [14]:
exec(final_code, globals())

sample = """date,product,quantity,price
2024-03-01,Widget,3,9.99
2024-03-02, widget ,2,9.99
2024-03-02,Gadget,1,24.50
2024-03-03,Gizmo,abc,5.00
2024-03-04,Gadget,-1,24.50
bad,row"""
print(summarize_sales(sample))

ALL TESTS PASSED
{'revenue_by_product': {'Widget': 49.95, 'Gadget': 24.5}, 'top_product': 'Widget', 'skipped_rows': 3}
